# One-domain .nas mesh

In [ ]:
# Read in data from swc file of a bifurcation and create vascularmd mesh for it
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pyvista as pv
from ArterialTree import ArterialTree

# ── Configuration ─────────────────────────────────────────────────────────────
data_folder_name = 'BG0014.CNG'
bifurcation_id   = 134
N = 64       # nodes around each cross-section circumference (must be multiple of 8)
d = 0.25     # longitudinal cross-section density
layer_ratio = [0.23, 0.15, 0.62]  # O-grid radius ratio [a, b, c], a+b+c = 1
num_a, num_b = 6, 6            # number of O-grid layers in parts a and b

swc_path = os.path.join('..', 'Data', data_folder_name, f'bifurcation_{bifurcation_id}_outer.swc')
out_dir  = os.path.join('..', 'Output')
os.makedirs(out_dir, exist_ok=True)

# Build the vascularmd (ArterialTree) model straight from the SWC centerline data
tree = ArterialTree("patient", "Aneurisk", swc_path)
tree.model_network()                               # parametric modeling of the network
tree.compute_cross_sections(N, d, parallel=False)   # mesh cross sections

volume_mesh = tree.mesh_volume(layer_ratio, num_a, num_b)
print(f"Meshed full network: {volume_mesh.n_points} points, {volume_mesh.n_cells} cells")

# ── Write the volume mesh to Nastran (.nas), tagging every element as one domain ──
# Native route (no VTK/pyvista hop): get the mesh as a meshio.Mesh built straight
# from the tree's internal arrays, then attach a per-cell property id via
# cell_data["nastran:ref"]. meshio's Nastran writer emits that value into the
# CHEXA PID field (field 3, blank by default). Every hexahedron gets PID 1
# ("fluid"), so the whole mesh reads as one explicitly-tagged solid domain that
# COMSOL recognises as a single selection. Nastran PIDs are integers, so the name
# "fluid" is just our label for PID 1.
mesh = tree.get_volume_mesh("meshio")               # meshio.Mesh(points, [("hexahedron", conn)])
n_cells = mesh.cells[0].data.shape[0]
FLUID_PID = 1
mesh.cell_data["nastran:ref"] = [np.full(n_cells, FLUID_PID, dtype=int)]

nas_path = os.path.join(out_dir, f'bifurcation_{bifurcation_id}_volume_mesh.nas')
mesh.write(nas_path)
print(f"Saved Nastran mesh -> {os.path.abspath(nas_path)}")
print(f"  all {n_cells} CHEXA tagged with PID {FLUID_PID} (= 'fluid'), one domain")


Loading swc file...
Modeling the network...
Meshing bifurcations.
Meshing edges.
Meshing volume...
Meshed full network: 69751 points, 66144 cells
Saved Nastran mesh -> /home/xbfl0349/arterial_network/Output/bifurcation_134_volume_mesh.nas
  all 66144 CHEXA tagged with PID 1 (= 'fluid'), one domain


# Two-domain .nas mesh

In [2]:
# Tag the layer_a (boundary) cells with PID 2 and everything else with PID 1.
#
# The volume mesh is a stack of "slabs": each cross-section pair contributes one
# hexahedron per 2-D O-grid quad, in the exact row order of ogrid_pattern_faces.
# So we (1) compute, per O-grid quad, whether it lies in part a (the radial band
# from the wall out to pb), (2) tile that flag across every slab, (3) write PID 2
# where layer_a else PID 1. Reuses the `tree` / mesh already built in the cell above.

# ── Stage 1: per-quad layer_a flag ────────────────────────────────────────────
# Mirror of ArterialTree.ogrid_pattern_faces (ArterialTree.py:2591), additionally
# recording each quad's radial index k (k = 0 at the wall, increasing inward -- the
# inner-loop variable at ArterialTree.py:2640). A quad is in part a when k <= num_a:
# lin_interp(surface, pb, num_a+2)[:-1] lays down num_a+1 intervals between the wall
# and pb, so the boundary band is num_a+1 cell layers thick. (Toggle to `k < num_a`
# for exactly num_a layers.)
def ogrid_faces_with_region(N, num_a, num_b):
    count = 0
    tot_faces = int(N/8) * (2 * (num_a + num_b + 2) + int(N/8)) * 4
    faces = np.zeros((tot_faces, 5), dtype=int)
    is_a  = np.zeros((tot_faces,), dtype=bool)
    shared_edge1 = []
    nb_faces = 0
    for s in range(4):
        j = 0
        for i in range(int(s * N/4), int((s + 1) * N/4)):
            if j <= int(N/8):
                if s != 0 and j == 0:
                    nb_vertices = int(num_a + num_b + 2)
                    ray_ind = list(range(count, count + nb_vertices)) + shared_edge1[::-1]
                    shared_edge1 = [ray_ind[-1]]
                elif s == 3:
                    nb_vertices = int(num_a + num_b + 2 + N/8)
                    ray_ind = list(range(count, count + nb_vertices)) + [first_ray[-j - 1]]
                else:
                    nb_vertices = int(num_a + num_b + 3 + N/8)
                    ray_ind = list(range(count, count + nb_vertices))
                    shared_edge1.append(ray_ind[-1])
                if j == int(N/8):
                    shared_edge2 = ray_ind[-int(N/8):]
            else:
                nb_vertices = num_a + num_b + 2
                ray_ind = list(range(count, count + nb_vertices)) + [shared_edge2[j - int(N/8) - 1]]
            if s == 0 and j == 0:
                first_ray = ray_ind
                ray_prec = ray_ind
            else:
                for k in range(min([len(ray_prec) - 1, len(ray_ind) - 1])):
                    faces[nb_faces, :] = np.array([4, ray_prec[k], ray_ind[k], ray_ind[k + 1], ray_prec[k + 1]])
                    is_a[nb_faces] = (k <= num_a)
                    nb_faces += 1
            count += nb_vertices
            ray_prec = ray_ind
            j = j + 1
    for k in range(min([len(ray_ind) - 1, len(first_ray) - 1])):
        faces[nb_faces, :] = np.array([4, ray_ind[k], first_ray[k], first_ray[k + 1], ray_ind[k + 1]])
        is_a[nb_faces] = (k <= num_a)
        nb_faces += 1
    return faces, is_a

faces, is_layer_a = ogrid_faces_with_region(N, num_a, num_b)
# Fidelity guard: the mirror must reproduce the library's own faces exactly,
# otherwise the radial-index labelling can't be trusted.
assert np.array_equal(faces, tree.ogrid_pattern_faces(N, num_a, num_b)), \
    "mirrored ogrid_pattern_faces diverged from ArterialTree.ogrid_pattern_faces"
nfaces = faces.shape[0]
print(f"O-grid quads per slab: {nfaces}  ->  {int(is_layer_a.sum())} layer_a, {int((~is_layer_a).sum())} rest")

# ── Stage 2: propagate the per-quad flag to every cell ────────────────────────
# Every slab writes exactly `nfaces` cells in this same quad order (vessel slabs
# and into-bifurcation slabs alike -- the bifurcation reorder only rewires the far
# side, not the row order/count). Validate against the link graph before tiling.
n_cells = tree.get_number_of_cells()
assert n_cells % nfaces == 0, f"cell count {n_cells} is not a multiple of {nfaces}"
n_slabs = n_cells // nfaces

link = tree.get_volume_link()                        # [edge_start, edge_end, crsec_index] per cell
link_slabs = link.reshape(n_slabs, nfaces, 3)
assert (link_slabs == link_slabs[:, :1, :]).all(), \
    "cells are not grouped into nfaces-sized slabs -- tiling would be misaligned"

cell_is_layer_a = np.tile(is_layer_a, n_slabs)
print(f"{n_cells} cells over {n_slabs} slabs -> "
      f"{int(cell_is_layer_a.sum())} layer_a cells (PID 2), {int((~cell_is_layer_a).sum())} rest (PID 1)")

# ── Stage 3: assign PID and write the two-domain .nas ─────────────────────────
LAYER_A_PID, REST_PID = 2, 1
pid = np.where(cell_is_layer_a, LAYER_A_PID, REST_PID).astype(int)

mesh = tree.get_volume_mesh("meshio")
mesh.cell_data["nastran:ref"] = [pid]                # -> CHEXA PID field (field 3)

nas_path = os.path.join(out_dir, f'bifurcation_{bifurcation_id}_volume_mesh_2domain.nas')
mesh.write(nas_path)
print(f"Saved two-domain Nastran mesh -> {os.path.abspath(nas_path)}")
print(f"  PID {LAYER_A_PID} = layer_a ({int(cell_is_layer_a.sum())} CHEXA), "
      f"PID {REST_PID} = rest ({int((~cell_is_layer_a).sum())} CHEXA)")


O-grid quads per slab: 624  ->  240 layer_a, 384 rest
66144 cells over 106 slabs -> 25440 layer_a cells (PID 2), 40704 rest (PID 1)
Saved two-domain Nastran mesh -> /home/xbfl0349/arterial_network/Output/bifurcation_134_volume_mesh_2domain.nas
  PID 2 = layer_a (25440 CHEXA), PID 1 = rest (40704 CHEXA)


In [3]:
# ── Verify the two-domain tagging ─────────────────────────────────────────────
import meshio

# 1) Parse the .nas directly: PID is field 3 (cols 17-24) of each CHEXA card.
pid_counts = {}
with open(nas_path) as fh:
    for ln in fh:
        if ln.startswith("CHEXA"):
            p = ln[16:24].strip()
            pid_counts[p] = pid_counts.get(p, 0) + 1
print("CHEXA PID counts in file:", pid_counts)
assert pid_counts.get("2", 0) == int(cell_is_layer_a.sum())
assert pid_counts.get("1", 0) == int((~cell_is_layer_a).sum())

# 2) meshio round-trip: the tags must survive a read-back and match what we wrote.
rt = meshio.read(nas_path)
ref = rt.cell_data["nastran:ref"][0]
print("round-trip unique PIDs:", np.unique(ref),
      "| PID2:", int((ref == 2).sum()), " PID1:", int((ref == 1).sum()))
assert set(np.unique(ref).tolist()) == {1, 2}
assert np.array_equal(ref, pid)

# 3) Geometric sanity: layer_a (PID 2) cells should sit at the OUTER radius.
#    Per slab, measure each cell-centroid's distance from that slab's mean centroid
#    (~ the section axis); layer_a cells should have the larger radius.
cent = tree.get_volume_mesh().cell_centers().points          # same order as `pid`
cent_slabs = cent.reshape(n_slabs, nfaces, 3)
r = np.linalg.norm(cent_slabs - cent_slabs.mean(axis=1, keepdims=True), axis=2)
is_a_slab = np.tile(is_layer_a, (n_slabs, 1))
print(f"median centroid radius:  layer_a = {np.median(r[is_a_slab]):.3f}   "
      f"rest = {np.median(r[~is_a_slab]):.3f}   (layer_a should be larger)")
assert np.median(r[is_a_slab]) > np.median(r[~is_a_slab])
print("All checks passed.")


CHEXA PID counts in file: {'2': 25440, '1': 40704}
round-trip unique PIDs: [1 2] | PID2: 25440  PID1: 40704
median centroid radius:  layer_a = 0.645   rest = 0.329   (layer_a should be larger)
All checks passed.
